# MC-Dropout U-Net — uncertainty + calibration (Task04 Hippocampus)

Uses a **preprocessed** dataset from the Kaggle input (no preprocessing here) and the **pretrained** baseline weights (the 5-dropout, CE-trained `mcdropout` model) from an input dataset. Compares calibration across four settings:

- **A. baseline (CE)** raw  vs  baseline + **post-hoc temperature T**
- **B. train-time calibrated** (`Dice + CE + λ·SB-ECE`, tag `mcdropout_cal`) raw  vs  + post-hoc T

**Before running:** commit & push all files (incl. `train_mc_recalibrate.py`, `loss_functions/calibration_loss.py`) to the repo cloned below. Set `DATA_IN` and `WEIGHTS_IN` to your input paths. Run top to bottom.

In [ ]:
# ===== init =====
import os

# (1) Preprocessed dataset: folder Task04_Hippocampus with preprocessed/ , splits.pkl , dataset.json
DATA_IN = "/kaggle/input/task04-hippocampus-prep/Task04_Hippocampus"   # <-- your preprocessed dataset
os.environ["DATA_DIR"] = DATA_IN          # config reads this -> no preprocessing needed
os.environ["TASK"]     = "Task04_Hippocampus"

# (2) Pretrained BASELINE weights: dir containing mcdropout_f<fold>_best.pth (5-dropout CE model)
WEIGHTS_IN = "/kaggle/input/mcdropout-weights"   # <-- your weights dataset

# hyper-params (MUST match how the baseline was trained; reused for the calibrated model)
FOLDS      = [0]        # e.g. [0, 1, 2, 3, 4] for full 5-fold CV
DROPOUT_P  = 0.4
MC_SAMPLES = 30
CAL_WEIGHT = 1.0       # lambda for SB-ECE in the train-time-calibrated model (try 1, 5, 10)
OUT_DIR    = "results" # weights / scores / uncertainty (inside the repo -> persists as output)

In [ ]:
# ===== repo + requirements =====
!rm -rf repo
!git clone "https://github.com/ThanoSnake/my_unet.git" repo
%cd repo
!pip install -q medpy batchgenerators==0.21

In [ ]:
# ===== setup =====
# (a) Kaggle ships HuggingFace 'datasets'; our local datasets/ needs __init__.py to win on sys.path.
import os
for d in ["datasets", "datasets/two_dim", "datasets/three_dim",
          "networks", "loss_functions", "evaluation", "utilities"]:
    os.makedirs(d, exist_ok=True)
    open(os.path.join(d, "__init__.py"), "a").close()

import torch, shutil, config
from networks.UNET_mc import MCDropoutUNet
_ = MCDropoutUNet(num_classes=config.NUM_CLASSES, in_channels=1, dropout_p=DROPOUT_P)(torch.rand(2, 1, 64, 64))

# (b) data comes straight from the read-only preprocessed input -> just verify it is there
assert config.PREPROCESSED_DIR.exists(), config.PREPROCESSED_DIR
assert config.SPLITS_FILE.exists(),      config.SPLITS_FILE
print("DATA_DIR:", config.DATA_DIR, "| classes:", config.NUM_CLASSES, "| labels:", config.LABELS)

# (c) bring the pretrained baseline weights into OUT_DIR so the scripts find them by default
os.makedirs(OUT_DIR, exist_ok=True)
for f in FOLDS:
    src = os.path.join(WEIGHTS_IN, f"mcdropout_f{f}_best.pth")
    dst = os.path.join(OUT_DIR, f"mcdropout_f{f}_best.pth")
    if os.path.exists(src):
        shutil.copy(src, dst); print("weights ready:", dst)
    else:
        print("WARNING: baseline weights not found:", src)

## A. Baseline model (CE-trained) — weights loaded, no training
Runs classic metrics, RAW uncertainty (T=1), then post-hoc temperature scaling (`_cal`).

In [ ]:
# ===== A1. baseline: classic metrics (dropout OFF) =====
for f in FOLDS:
    !python test_mc.py --fold {f} --tag mcdropout --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}

In [ ]:
# ===== A2. baseline: RAW uncertainty (T=1) + foreground calibration =====
for f in FOLDS:
    !python uncertainty_mc.py --fold {f} --tag mcdropout --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --temperature 1.0 --save-volumes

In [ ]:
# ===== A3. baseline: post-hoc temperature scaling -> '_cal' outputs =====
for f in FOLDS:
    !python calibrate_mc.py   --fold {f} --tag mcdropout --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}
    !python uncertainty_mc.py --fold {f} --tag mcdropout --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --save-volumes

## B. Train-time calibrated model (`Dice + CE + λ·SB-ECE`)
No pretrained weights for this one — we train it (tag `mcdropout_cal`). Goal: lower RAW ECE than the baseline, and a residual post-hoc T closer to 1.

In [ ]:
# ===== B1. train the calibrated model (early stopping, saves best) =====
# --num-workers 0 avoids the CUDA-in-forked-worker crash on Kaggle.
for f in FOLDS:
    !python train_mc_recalibrate.py --fold {f} --dropout-p {DROPOUT_P} --cal-weight {CAL_WEIGHT} --out-dir {OUT_DIR} --num-workers 0

In [ ]:
# ===== B2. calibrated model: classic metrics (check Dice did NOT drop) =====
for f in FOLDS:
    !python test_mc.py --fold {f} --tag mcdropout_cal --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}

In [ ]:
# ===== B3. calibrated model: RAW uncertainty (T=1) -> native calibration =====
for f in FOLDS:
    !python uncertainty_mc.py --fold {f} --tag mcdropout_cal --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --temperature 1.0 --save-volumes

In [ ]:
# ===== B4. calibrated model + post-hoc T (T should be closer to 1 than the baseline's) =====
for f in FOLDS:
    !python calibrate_mc.py   --fold {f} --tag mcdropout_cal --dropout-p {DROPOUT_P} --out-dir {OUT_DIR}
    !python uncertainty_mc.py --fold {f} --tag mcdropout_cal --dropout-p {DROPOUT_P} --mc-samples {MC_SAMPLES} --out-dir {OUT_DIR} --save-volumes

## Compare
Foreground / macro-per-class ECE and the temperature T across the four settings.

In [ ]:
# ===== compare foreground ECE (and T) across the four settings =====
import os, json

def load_ece(path):
    if not os.path.exists(path):
        return None
    d = json.load(open(path)); c = d["calibration"]
    return c["foreground_ece"], c["macro_foreground_ece"], d.get("temperature", 1.0)

unc = os.path.join(OUT_DIR, "uncertainty")
print(f"{'setting':<24}{'fg_ECE':>9}{'macro_ECE':>11}{'T':>7}")
for f in FOLDS:
    for label, fn in [
        (f"f{f} baseline raw",     f"mcdropout_f{f}_uncertainty.json"),
        (f"f{f} baseline +T",      f"mcdropout_f{f}_cal_uncertainty.json"),
        (f"f{f} calibrated raw",   f"mcdropout_cal_f{f}_uncertainty.json"),
        (f"f{f} calibrated +T",    f"mcdropout_cal_f{f}_cal_uncertainty.json"),
    ]:
        r = load_ece(os.path.join(unc, fn))
        if r:
            print(f"{label:<24}{r[0]:>9.4f}{r[1]:>11.4f}{r[2]:>7.3f}")

In [ ]:
# ===== show the calibration diagrams for the four settings =====
import os, glob
from IPython.display import Image, display

unc = os.path.join(OUT_DIR, "uncertainty")
for f in FOLDS:
    for label, fn in [
        ("baseline raw",   f"mcdropout_f{f}_calibration.png"),
        ("baseline +T",    f"mcdropout_f{f}_cal_calibration.png"),
        ("calibrated raw", f"mcdropout_cal_f{f}_calibration.png"),
        ("calibrated +T",  f"mcdropout_cal_f{f}_cal_calibration.png"),
    ]:
        p = os.path.join(unc, fn)
        if os.path.exists(p):
            print(f"fold {f} — {label}: {fn}")
            display(Image(p))